# convT-as-flipped-padded-conv — faded example 3: Fill the hand-computed ConvT output for a 2x2 example

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `convT-as-flipped-padded-conv`. Running the beacon reports progress on the `CNN: ConvT as flipped padded conv` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT as flipped padded conv` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`convT-as-flipped-padded-conv`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "convT-as-flipped-padded-conv"
DD_SUBTOPIC = "CNN: ConvT as flipped padded conv"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

On a small integer example the ConvT-as-flipped-padded-conv identity is byte-exact. ConvT scatters a scaled copy of the kernel at every input position and sums the overlaps. Reconstructing the expected output matrix by hand cements *why* the flip is needed: a non-symmetric kernel deposits its taps in a specific direction.

## Faded exercise 3

Given `x = [[2, 0], [1, 3]]` (shape `(1,1,2,2)`) and ConvT-layout kernel `w = [[1, 0], [2, 0]]` (shape `(1,1,2,2)`), the code computes both `F.conv_transpose2d(x, w)` and the flipped-padded-conv2d equivalent. You must fill in `expected`: the `(1,1,3,3)` matrix obtained by summing the four kernel placements by hand. The test checks all three agree exactly.

**Fill in:** the (1,1,3,3) ground-truth ConvT output tensor formed by scattering w scaled by each input value and summing overlaps: [[2,0,0],[5,3,0],[2,6,0]]

In [ ]:
import torch.nn.functional as F

x = t.tensor([[[[2., 0.], [1., 3.]]]])
w = t.tensor([[[[1., 0.], [2., 0.]]]])   # ConvT layout (IC=1, OC=1, KH=2, KW=2)

convT_out = F.conv_transpose2d(x, w)
x_pad = F.pad(x, (1, 1, 1, 1))
w_eq = w.flip([2, 3]).transpose(0, 1).contiguous()
conv_equiv = F.conv2d(x_pad, w_eq)

expected = None  # TODO: the (1,1,3,3) hand-summed ConvT output tensor

print('convT == expected:', t.equal(convT_out, expected))
print('conv_equiv == expected:', t.equal(conv_equiv, expected))


def _test():
    import torch.nn.functional as F
    x = t.tensor([[[[2., 0.], [1., 3.]]]])
    w = t.tensor([[[[1., 0.], [2., 0.]]]])
    convT_out = F.conv_transpose2d(x, w)
    x_pad = F.pad(x, (1, 1, 1, 1))
    w_eq = w.flip([2, 3]).transpose(0, 1).contiguous()
    conv_equiv = F.conv2d(x_pad, w_eq)
    assert expected.shape == (1, 1, 3, 3), expected.shape
    assert t.equal(convT_out, expected), convT_out
    assert t.equal(conv_equiv, expected), conv_equiv


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn.functional as F

x = t.tensor([[[[2., 0.], [1., 3.]]]])
w = t.tensor([[[[1., 0.], [2., 0.]]]])   # ConvT layout (IC=1, OC=1, KH=2, KW=2)

convT_out = F.conv_transpose2d(x, w)
x_pad = F.pad(x, (1, 1, 1, 1))
w_eq = w.flip([2, 3]).transpose(0, 1).contiguous()
conv_equiv = F.conv2d(x_pad, w_eq)

expected = t.tensor([[[[2., 0., 0.],
                       [5., 3., 0.],
                       [2., 6., 0.]]]])

print('convT == expected:', t.equal(convT_out, expected))
print('conv_equiv == expected:', t.equal(conv_equiv, expected))
```
</details>